# Week 12 — RL for Ranking and Two-Sided Matching · Workbook

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # for common/
sys.path.insert(0, os.path.abspath('.'))    # for project/
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
from project import solutions as sol


## Exercise 1 — Jaccard similarity (★)

Implement Jaccard similarity between two skill sets:
$J(A, B) = |A \cap B| / |A \cup B|$, with the convention $J(\emptyset, \emptyset) = 1$.

> **Hint**: use Python `set` operations.

In [ ]:
ans = sol.exercise_1_jaccard(['python', 'sql', 'ml'], ['python', 'ml', 'spark'])
print('Jaccard =', round(ans, 3))
assert abs(ans - 0.5) < 1e-9


## Exercise 2 — Position-bias model (★)

Return the position-bias vector $p_k = 1/k^{\eta}$ for positions $k=1, \ldots, K$.

> **Hint**: $p_1 = 1$ by convention.

In [ ]:
pb = sol.exercise_2_position_bias(K=10, eta=1.0)
print('p_k =', np.round(pb, 3))
assert pb[0] == 1.0
assert pb[-1] < pb[0]


## Exercise 3 — Two-tower NDCG (★★)

Train the two-tower retriever on the bundled ATS dataset (200 cands, 50
jobs). Report NDCG@10 and NDCG@5 against the ground-truth Jaccard
relevance.

> **Hint**: see `project.two_tower.train_two_tower`. NDCG@10 should be > 0.5 for a small-but-honest seed.

In [ ]:
n10, n5 = sol.exercise_3_two_tower_ndcg(seed=0, epochs=40)
print(f'NDCG@10 = {n10:.3f}, NDCG@5 = {n5:.3f}')
assert n10 > 0.45 and n5 > 0.45


## Exercise 4 — IPS-LTR vs.\ naive click count (★★)

Simulate biased clicks from a known relevance vector; compute the naive
click sum vs.\ the IPS-LTR value for a target ranking.

> **Hint**: position bias $p_k = 1/k$. Each click at position $k$ in the logged ranking contributes $p_{k'} / p_k$ to the target value, where $k'$ is the item's position in the target.

In [ ]:
naive, ips = sol.exercise_4_ips_ltr(seed=0)
print('naive click sum =', round(naive, 2))
print('IPS-LTR value   =', round(ips, 2))


## Exercise 5 — LinTS reranker NDCG (★★)

Train the two-tower, then a LinTS reranker on simulated hire feedback.
Report `tt_ndcg@10` and `rerank_ndcg@10`. The reranker may not always
beat the retriever on this tiny dataset (LinTS posterior is wide); the
**point** is to inspect the pipeline.

> **Hint**: the pipeline lives in `project.train.train`.

In [ ]:
tt, rr = sol.exercise_5_reranker_improvement(seed=0, epochs=30)
print(f'tt NDCG@10 = {tt:.3f}, rerank NDCG@10 = {rr:.3f}')
assert tt > 0.4


## Exercise 6 — Demographic-parity gap (★★)

Audit the two-tower top-10 for demographic-parity: what fraction of top-10
slots is filled by group F vs.\ group M? Return the absolute gap, averaged
over jobs.

> **Hint**: see `project.fairness.demographic_parity_gap`. The synthetic dataset has a gender-skill bias (women under-represent leadership) that creates a non-zero gap even for a 'fair' model.

In [ ]:
gap = sol.exercise_6_dp_gap(seed=0, epochs=20)
print(f'DP gap = {gap:.3f}')
assert 0.0 <= gap <= 1.0


## Exercise 7 — Equalised-opportunity gap (★★)

Same audit but conditioning on *qualified* candidates (true match $\geq 0.4$).

> **Hint**: EO is typically much smaller than DP when the model genuinely tracks skill.

In [ ]:
gap = sol.exercise_7_eo_gap(seed=0, epochs=20)
print(f'EO gap = {gap:.3f}')
assert 0.0 <= gap <= 1.0


## Exercise 8 — Calibration ECE (★★)

Synthetic over-confident scores: $s \sim \text{Uniform}[0,1]$,
$P(y=1) = s^2$. Fit Platt, isotonic, Beta calibrators on a 500-sample
train split; report ECE on the held-out 500.

> **Hint**: all three calibrators should improve over the raw ECE; isotonic typically wins on $n \geq 1000$.

In [ ]:
eces = sol.exercise_8_calibration(seed=0)
for name, val in eces.items():
    print(f'  {name:10s} ECE = {val:.4f}')
assert eces['beta'] < eces['raw']
assert eces['isotonic'] < eces['raw']


## Exercise 9 — Exposure by group (★★)

Position-bias-weighted exposure, averaged across jobs, separated by group.
The two numbers should be similar (within ~10%) for a fair ranker.

> **Hint**: use $p_k = 1/k$.

In [ ]:
exp_f, exp_m = sol.exercise_9_exposure(seed=0, epochs=20)
print(f'exposure F = {exp_f:.4f},  exposure M = {exp_m:.4f}')
print(f'ratio = {exp_f / max(exp_m, 1e-9):.3f}')
assert exp_f > 0 and exp_m > 0


## Exercise 10 — ATS recommendation memo (★★★, open-ended, *meaningful objective*)

Run the full pipeline. Return a dictionary with NDCG@10 (pre/post), DP gap
(pre/post), EO gap (pre/post), and ECE (raw/beta). Write a one-page
recommendation memo (markdown cell below) for the synthetic ATS deploy team
arguing for or against shipping this model.


> **Hint**: the memo should reference at least three of the safety failure modes from the lecture (proxy discrimination, calibration drift, position-bias mis-modelling, supply-side equilibrium).

In [ ]:
memo = sol.exercise_10_ats_recommendation_memo(seed=0, epochs=30)
for k, v in memo.items():
    print(f'  {k:20s} = {v:.4f}')


## Recommendation memo (write your own here)

**Subject:** Recommendation on the v2 candidate-ranker deploy

**Summary:** ... (one paragraph)

**Strengths:**
- NDCG@10 above baseline.
- [Insert measured held-out calibration ECE and its evaluation split.]

**Concerns:**
- [Insert the measured top-k representation gap; it is not a selection-rate DP gap.]
- LinTS posterior wide on cold requisitions (hierarchical-TS recommended).
- ...

**Recommendation:** [ship / hold / ship-with-A-B / fail-the-deploy]


## Wrap-up

You have built:
1. A two-tower retriever (in-batch-negatives InfoNCE).
2. A LinTS contextual-bandit reranker.
3. A position-bias-aware CLTR estimator (IPS-LTR).
4. A multi-dimensional fairness audit (DP, EO, exposure).
5. Three calibrators (Platt, Isotonic, Beta).

Next week: multi-agent RL and AlphaZero --- ranking gives way to games.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 — Jaccard similarity (★)</summary>

> **Hint**: use Python `set` operations.

</details>

<details>
<summary>Exercise 2 — Position-bias model (★)</summary>

> **Hint**: $p_1 = 1$ by convention.

</details>

<details>
<summary>Exercise 3 — Two-tower NDCG (★★)</summary>

> **Hint**: see `project.two_tower.train_two_tower`. NDCG@10 should be > 0.5 for a small-but-honest seed.

</details>

<details>
<summary>Exercise 4 — IPS-LTR vs.\ naive click count (★★)</summary>

> **Hint**: position bias $p_k = 1/k$. Each click at position $k$ in the logged ranking contributes $p_{k'} / p_k$ to the target value, where $k'$ is the item's position in the target.

</details>

<details>
<summary>Exercise 5 — LinTS reranker NDCG (★★)</summary>

> **Hint**: the pipeline lives in `project.train.train`.

</details>

<details>
<summary>Exercise 6 — Demographic-parity gap (★★)</summary>

> **Hint**: see `project.fairness.demographic_parity_gap`. The synthetic dataset has a gender-skill bias (women under-represent leadership) that creates a non-zero gap even for a 'fair' model.

</details>

<details>
<summary>Exercise 7 — Equalised-opportunity gap (★★)</summary>

> **Hint**: EO is typically much smaller than DP when the model genuinely tracks skill.

</details>

<details>
<summary>Exercise 8 — Calibration ECE (★★)</summary>

> **Hint**: all three calibrators should improve over the raw ECE; isotonic typically wins on $n \geq 1000$.

</details>

<details>
<summary>Exercise 9 — Exposure by group (★★)</summary>

> **Hint**: use $p_k = 1/k$.

</details>

<details>
<summary>Exercise 10 — ATS recommendation memo (★★★, open-ended, *meaningful objective*)</summary>

> **Hint**: the memo should reference at least three of the safety failure modes from the lecture (proxy discrimination, calibration drift, position-bias mis-modelling, supply-side equilibrium).

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
